In [8]:
# ============================================================
# Pregame PA Model Training
# Predicts PA outcome based on pitcher vs hitter matchup
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

from catboost import CatBoostClassifier

In [9]:
# ----------------------------
# Load Statcast dataset
# ----------------------------

DATA_PATH = "data/statcast_2021_2025.parquet"

df = pd.read_parquet(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 3846144
Columns: 119


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,FF,2021-11-02,93.7,1.39,6.72,"Smith, Will",493329,519293,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,5,"Yuli Gurriel grounds out, shortstop Dansby Swa...",W,R,L,HOU,ATL,X,6,ground_ball,0,2,2021,0.57,1.21,0.190973,2.68051,<NA>,<NA>,488726,2,9,Bot,98.12,136.43,<NA>,<NA>,<NA>,<NA>,-4.348927,-136.263269,-7.335202,8.101619,29.384843,-15.743652,3.37,1.53,5,93.8,-28,93.4,2112,6.1,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.39,0.074,0.068,0.0,1,0,0,2,69,3,4-Seam Fastball,0,7,0,7,7,0,0,7,Standard,Standard,146,-0.001,-0.164,<NA>,<NA>,0.077,0.164,93.8,-7,-7,0.001,0.001,31,37,32,37,1,3,3,2,<NA>,<NA>,1.39,0.57,-0.57,46.9,<NA>,<NA>,<NA>,<NA>,<NA>,2021
1,FF,2021-11-02,92.9,1.38,6.72,"Smith, Will",493329,519293,None,foul,<NA>,<NA>,<NA>,<NA>,5,Foul,W,R,L,HOU,ATL,S,<NA>,None,0,1,2021,0.9,1.34,0.01112,2.117733,<NA>,<NA>,488726,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-5.455433,-134.989926,-8.881689,12.188852,30.69001,-14.009571,3.37,1.53,156,75.0,15,92.6,2206,6.3,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.22,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,69,2,4-Seam Fastball,0,7,0,7,7,0,0,7,Standard,Standard,145,0.0,-0.056,<NA>,<NA>,<NA>,0.056,88.0,-7,-7,0.001,0.001,31,37,32,37,1,3,3,2,<NA>,<NA>,1.32,0.9,-0.9,48.0,<NA>,<NA>,<NA>,<NA>,<NA>,2021
2,FF,2021-11-02,93.1,1.35,6.73,"Smith, Will",493329,519293,None,called_strike,<NA>,<NA>,<NA>,<NA>,6,Called Strike,W,R,L,HOU,ATL,S,<NA>,None,0,0,2021,0.81,1.52,0.782772,2.133925,<NA>,<NA>,488726,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-3.230974,-135.201801,-9.255781,10.67848,31.699974,-11.7058,3.4,1.53,<NA>,<NA>,<NA>,92.5,2216,6.2,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,69,1,4-Seam Fastball,0,7,0,7,7,0,0,7,Standard,Standard,143,0.0,-0.049,<NA>,<NA>,<NA>,0.049,<NA>,-7,-7,0.001,0.001,31,37,32,37,1,3,3,2,<NA>,<NA>,1.14,0.81,-0.81,46.7,<NA>,<NA>,<NA>,<NA>,<NA>,2021
3,FF,2021-11-02,94.6,1.31,6.73,"Smith, Will",670541,519293,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,5,Yordan Alvarez flies out to left fielder Eddie...,W,L,L,HOU,ATL,X,7,fly_ball,3,2,2021,0.85,1.27,-0.230264,2.658489,<NA>,<NA>,488726,1,9,Bot,68.5,86.79,<NA>,<NA>,<NA>,<NA>,-5.901934,-137.422092,-7.652311,12.118242,35.102245,-14.577857,3.58,1.68,312,92.7,39,93.8,2263,6.3,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.25,0.025,0.041,0.0,1,0,0,3,68,6,4-Seam Fastball,0,7,0,7,7,0,0,7,Infield shi

In [10]:
# ----------------------------
# Plate appearance outcomes
# ----------------------------

pa_events = [
    "single",
    "double",
    "triple",
    "home_run",
    "walk",
    "strikeout",
    "field_out",
    "force_out",
    "grounded_into_double_play",
    "sac_fly"
]

df = df[df["events"].isin(pa_events)].copy()

print("PA rows:", df.shape)

PA rows: (972893, 119)


In [11]:
# ----------------------------
# Simplify PA outcome classes
# ----------------------------

def simplify_event(x):

    if x in ["single"]:
        return "single"

    if x in ["double"]:
        return "double"

    if x in ["triple"]:
        return "triple"

    if x in ["home_run"]:
        return "hr"

    if x in ["walk"]:
        return "walk"

    if x in ["strikeout"]:
        return "strikeout"

    return "out"


df["pa_outcome"] = df["events"].apply(simplify_event)

df["pa_outcome"].value_counts()

pa_outcome
out          443107
strikeout    227835
single       141595
walk          81979
double        43720
hr            30918
triple         3739
Name: count, dtype: int64

In [12]:
# ----------------------------
# Pregame features
# ----------------------------

features = [
    "pitcher",
    "batter",
    "p_throws",
    "stand",
    "inning",
    "outs_when_up",
    "balls",
    "strikes",
    "on_1b",
    "on_2b",
    "on_3b"
]

df_model = df[features + ["pa_outcome"]].dropna()

print("Training rows:", df_model.shape)

Training rows: (22449, 12)


In [13]:
# ----------------------------
# Train model
# ----------------------------

X = df_model[features]
y = df_model["pa_outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    verbose=50,
    task_type="GPU"
)

model.fit(
    X_train,
    y_train,
    cat_features=[0,1,2,3]
)

0:	learn: 1.8558559	total: 49.3ms	remaining: 14.8s
50:	learn: 1.1203098	total: 832ms	remaining: 4.06s
100:	learn: 1.0764813	total: 1.64s	remaining: 3.22s
150:	learn: 1.0550617	total: 2.46s	remaining: 2.43s
200:	learn: 1.0406701	total: 3.25s	remaining: 1.6s
250:	learn: 1.0259543	total: 4.04s	remaining: 789ms
299:	learn: 1.0114822	total: 4.8s	remaining: 0us


CatBoostClassifier(depth=6, iterations=300, learning_rate=0.05, loss_function='MultiClass', task_type='GPU', verbose=50)

In [14]:
# ----------------------------
# Evaluate
# ----------------------------

pred = model.predict_proba(X_test)

loss = log_loss(y_test, pred)

print("Log Loss:", loss)

Log Loss: 1.1217431947030916


In [15]:
# ----------------------------
# Save model
# ----------------------------

model.save_model("models/pregame_pa_model.cbm")

print("Pregame model saved")

Pregame model saved


In [16]:
import pandas as pd

pred_labels = model.predict(X_test)

acc = (pred_labels.flatten() == y_test).mean()

print("Accuracy:", acc)

Accuracy: 0.534521158129176
